In [2]:
# !pip install ctgan sdv pandas numpy

import pandas as pd
import numpy as np
from ctgan import CTGAN

# ---------------------------------------------------------
# 1. LOAD DỮ LIỆU & CẤU HÌNH
# ---------------------------------------------------------
print("Đang đọc dữ liệu...")
data = pd.read_csv('Agri_Data_Cleaned.csv')

discrete_columns = [
    'District', 'Season', 'Crop Name', 'Transplant', 
    'Growth', 'Harvest', 'pH_Suitability', 
    'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'Extreme_Heat_Risk', 'Is_Extreme_Heat',
    'is_extreme_Heat_Stress_Days', 
    'is_extreme_Wind_Max'
]

# ---------------------------------------------------------
# 2. HUẤN LUYỆN CTGAN
# ---------------------------------------------------------
print("Đang khởi tạo và huấn luyện CTGAN (Epochs=300)...")
ctgan = CTGAN(epochs=300, batch_size=500, verbose=True)
ctgan.fit(data, discrete_columns)

# ---------------------------------------------------------
# 3. SINH DỮ LIỆU
# ---------------------------------------------------------
print("Đang sinh dữ liệu mới...")
num_rows = 20000
synthetic_data = ctgan.sample(num_rows)

print("Đang thực hiện Hậu kiểm Logic (Logic Enforcement)...")

# =========================================================
# 4. HẬU XỬ LÝ TOÀN DIỆN (COMPREHENSIVE POST-PROCESSING)
# =========================================================

# --- HELPER: Hàm an toàn chia cho 0 ---
def safe_div_robust(numerator, denominator, epsilon=1e-6):
    denom_safe = denominator.copy()
    mask_too_small = np.abs(denom_safe) < epsilon
    denom_safe[mask_too_small] = np.sign(denom_safe[mask_too_small]) * epsilon
    denom_safe[denom_safe == 0] = epsilon
    return numerator / denom_safe

# --- A. CHẶN BIÊN ÂM ---
non_negative_cols = [
    'Rainfall', 'Soil_Moisture_mm', 'Avg_Salinity_Index', 
    'Organic_Carbon', 'Nitrogen', 'sm_surface', 'sm_rootzone',
    'Wind_Mean', 'Wind_Max', 'Heat_Stress_Days',
    'EVI', 'LAI', 'FPAR' 
]
for col in non_negative_cols:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(lower=0)


# --- B. LOGIC CƠ BẢN (MIN/MEAN/MAX) ---
def enforce_min_mean_max(df, col_min, col_mean, col_max):
    if {col_min, col_mean, col_max}.issubset(df.columns):
        mask_wrong = df[col_min] > df[col_max]
        cols = [col_min, col_max]
        df.loc[mask_wrong, cols] = df.loc[mask_wrong, cols].values[:, ::-1]
        df[col_mean] = df[col_mean].clip(lower=df[col_min], upper=df[col_max])
    return df

synthetic_data = enforce_min_mean_max(synthetic_data, 'Min Temp', 'Avg Temp', 'Max Temp')
synthetic_data = enforce_min_mean_max(synthetic_data, 'Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity')
synthetic_data = enforce_min_mean_max(synthetic_data, 'NDVI_Season_Min', 'NDVI_Season_Mean', 'NDVI_Season_Max')

for col in ['Min Relative Humidity', 'Avg Humidity', 'Max Relative Humidity']:
    if col in synthetic_data.columns:
        synthetic_data[col] = synthetic_data[col].clip(0, 100).round(1)

ndvi_cols = [c for c in synthetic_data.columns if 'NDVI' in c]
for col in ndvi_cols:
    synthetic_data[col] = synthetic_data[col].clip(-1.0, 1.0)


# --- CHUẨN HÓA CẢM BIẾN KHÍ TƯỢNG (SENSOR PRECISION) ---
weather_cols = ['Min Temp', 'Avg Temp', 'Max Temp']
if all(col in synthetic_data.columns for col in weather_cols):
    synthetic_data[weather_cols] = synthetic_data[weather_cols].round(1)

if 'Rainfall' in synthetic_data.columns:
    synthetic_data['Rainfall'] = synthetic_data['Rainfall'].round(2)


# --- C. ĐỒNG BỘ CÁC CỘT TỈ LỆ ---
if 'Rain_Temp_Ratio' in synthetic_data.columns:
    temp_safe = synthetic_data['Avg Temp'].copy()
    mask_cold = temp_safe <= 0
    ratio = safe_div_robust(synthetic_data['Rainfall'], temp_safe)
    ratio[mask_cold] = 0 
    synthetic_data['Rain_Temp_Ratio'] = ratio 

if 'CN_Ratio' in synthetic_data.columns:
    synthetic_data['Nitrogen'] = synthetic_data['Nitrogen'].clip(lower=0.001)
    synthetic_data['CN_Ratio'] = (synthetic_data['Organic_Carbon'] / synthetic_data['Nitrogen'])

if 'Moisture_Ratio' in synthetic_data.columns:
    synthetic_data['sm_surface'] = synthetic_data['sm_surface'].clip(lower=0.001)
    synthetic_data['Moisture_Ratio'] = (synthetic_data['sm_rootzone'] / synthetic_data['sm_surface'])


# --- D. XỬ LÝ NDVI (CV & RANGE) ---
if 'NDVI_Season_Range' in synthetic_data.columns:
    synthetic_data['NDVI_Season_Range'] = (synthetic_data['NDVI_Season_Max'] - synthetic_data['NDVI_Season_Min'])

if 'NDVI_Season_CV' in synthetic_data.columns:
    synthetic_data['NDVI_Season_Std'] = synthetic_data['NDVI_Season_Std'].clip(lower=0)
    mean_abs = synthetic_data['NDVI_Season_Mean'].abs()
    synthetic_data['NDVI_Season_CV'] = safe_div_robust(synthetic_data['NDVI_Season_Std'], mean_abs)


# --- E. LOGIC CỜ & RỦI RO ---

# 1. Heat Stress Days Flag
THRESHOLD_HEAT_DAYS = 43.0 
if 'is_extreme_Heat_Stress_Days' in synthetic_data.columns and 'Heat_Stress_Days' in synthetic_data.columns:
    mask_extreme = synthetic_data['is_extreme_Heat_Stress_Days'] == 1
    mask_update = mask_extreme & (synthetic_data['Heat_Stress_Days'] < THRESHOLD_HEAT_DAYS)
    
    # Nhóm Cực đoan: Random từ 43.0 trở lên
    synthetic_data.loc[mask_update, 'Heat_Stress_Days'] = np.random.uniform(THRESHOLD_HEAT_DAYS, 65.0, size=mask_update.sum())
    
    mask_normal = synthetic_data['is_extreme_Heat_Stress_Days'] == 0
    mask_update_normal = mask_normal & (synthetic_data['Heat_Stress_Days'] >= THRESHOLD_HEAT_DAYS)
    
    # [FIX ĐIỂM KỲ DỊ 1] Sự phản bội của phép làm tròn
    # Hạ trần xuống 42.4. Vì 42.4 * 2 = 84.8 -> round = 85 -> chia 2 = 42.5. (An toàn < 43.0)
    # Nếu để 42.9: 42.9 * 2 = 85.8 -> round = 86 -> chia 2 = 43.0 (LỖI!)
    synthetic_data.loc[mask_update_normal, 'Heat_Stress_Days'] = np.random.uniform(0.0, 42.4, size=mask_update_normal.sum())

    # Format toàn bộ cột để đồng nhất chữ ký dữ liệu
    synthetic_data['Heat_Stress_Days'] = (synthetic_data['Heat_Stress_Days'] * 2).round(0) / 2

# 2. Wind Logic
THRESHOLD_WIND = 11.0
if 'is_extreme_Wind_Max' in synthetic_data.columns and 'Wind_Max' in synthetic_data.columns:
    mask_extreme_wind = synthetic_data['is_extreme_Wind_Max'] == 1
    mask_update = mask_extreme_wind & (synthetic_data['Wind_Max'] < THRESHOLD_WIND)
    synthetic_data.loc[mask_update, 'Wind_Max'] = np.random.uniform(THRESHOLD_WIND, 35.0, size=mask_update.sum())
    
    mask_normal_wind = synthetic_data['is_extreme_Wind_Max'] == 0
    mask_update_normal = mask_normal_wind & (synthetic_data['Wind_Max'] >= THRESHOLD_WIND)
    synthetic_data.loc[mask_update_normal, 'Wind_Max'] = np.random.uniform(2.0, 10.0, size=mask_update_normal.sum())
    
    if 'Wind_Mean' in synthetic_data.columns:
         mask_wrong_wind = synthetic_data['Wind_Mean'] > synthetic_data['Wind_Max']
         synthetic_data.loc[mask_wrong_wind, 'Wind_Mean'] = synthetic_data.loc[mask_wrong_wind, 'Wind_Max'] * 0.8
    
    synthetic_data['Wind_Max'] = synthetic_data['Wind_Max'].round(2)
    if 'Wind_Mean' in synthetic_data.columns:
        synthetic_data['Wind_Mean'] = synthetic_data['Wind_Mean'].round(2)


# 3. Label-Flag Correlation
if 'Extreme_Heat_Risk' in synthetic_data.columns and 'Is_Extreme_Heat' in synthetic_data.columns:
    mask_contradiction = (synthetic_data['Is_Extreme_Heat'] == 1) & (synthetic_data['Extreme_Heat_Risk'] == 'Low Risk')
    if mask_contradiction.any():
        synthetic_data.loc[mask_contradiction, 'Extreme_Heat_Risk'] = 'High Risk' 

    mask_low_risk = synthetic_data['Extreme_Heat_Risk'] == 'Low Risk'
    synthetic_data.loc[mask_low_risk, 'Is_Extreme_Heat'] = 0


# --- F. CÁC FIX CƠ BẢN KHÁC ---
synthetic_data['Area'] = synthetic_data['Area'].clip(lower=1).round(0).astype(int)

# Yield logic
synthetic_data['Yield'] = synthetic_data['Yield'].clip(lower=0)
synthetic_data['Production'] = (synthetic_data['Area'] * synthetic_data['Yield']).round(0).astype(int)
synthetic_data['Yield'] = synthetic_data['Production'] / synthetic_data['Area'] 


# Soil Logic
soil_cols = ['Sand', 'Silt', 'Clay']
if all(col in synthetic_data.columns for col in soil_cols):
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)
    
    total_soil = synthetic_data[soil_cols].sum(axis=1)
    mask_zero_sum = total_soil == 0
    if mask_zero_sum.any():
        synthetic_data.loc[mask_zero_sum, 'Sand'] = 33.33
        synthetic_data.loc[mask_zero_sum, 'Silt'] = 33.33
        synthetic_data.loc[mask_zero_sum, 'Clay'] = 33.34
        total_soil[mask_zero_sum] = 100.0
        
    for col in soil_cols:
        synthetic_data[col] = (synthetic_data[col] / total_soil * 100)
        
    synthetic_data['Sand'] = synthetic_data['Sand'].round(2)
    synthetic_data['Silt'] = synthetic_data['Silt'].round(2)
    
    synthetic_data['Clay'] = (100.00 - synthetic_data['Sand'] - synthetic_data['Silt']).round(2)
    
    # [FIX ĐIỂM KỲ DỊ 2] Smart Soil Balancer
    # Thay vì cộng mù quáng vào Sand, ta chọn cột lớn nhất (Sand hoặc Silt) để gánh sai số
    mask_neg_clay = synthetic_data['Clay'] < 0
    
    # Case 1: Clay âm và Sand >= Silt -> Trừ bớt ở Sand (Cộng số âm Clay vào Sand)
    mask_sand_dest = mask_neg_clay & (synthetic_data['Sand'] >= synthetic_data['Silt'])
    if mask_sand_dest.any():
        synthetic_data.loc[mask_sand_dest, 'Sand'] += synthetic_data.loc[mask_sand_dest, 'Clay']
    
    # Case 2: Clay âm và Silt > Sand -> Trừ bớt ở Silt (Cộng số âm Clay vào Silt)
    mask_silt_dest = mask_neg_clay & (synthetic_data['Sand'] < synthetic_data['Silt'])
    if mask_silt_dest.any():
        synthetic_data.loc[mask_silt_dest, 'Silt'] += synthetic_data.loc[mask_silt_dest, 'Clay']

    # Cuối cùng đưa Clay về 0 và Clip toàn bộ để đảm bảo an toàn tuyệt đối
    if mask_neg_clay.any():
        synthetic_data.loc[mask_neg_clay, 'Clay'] = 0
        
    synthetic_data[soil_cols] = synthetic_data[soil_cols].clip(lower=0)

if 'Rootzone_Surface_Diff' in synthetic_data.columns:
    synthetic_data['Rootzone_Surface_Diff'] = (synthetic_data['sm_rootzone'] - synthetic_data['sm_surface'])

if 'pH' in synthetic_data.columns:
    synthetic_data['pH'] = synthetic_data['pH'].clip(3.5, 9.0).round(2)

# ---------------------------------------------------------
# 5. LƯU FILE
# ---------------------------------------------------------
output_file = 'Agri_Data_CTGAN_20k_Flawless.csv'
synthetic_data.to_csv(output_file, index=False)
print(f"Hoàn tất! Dữ liệu đã được lưu tại: {output_file}")
print("Phiên bản Flawless: Đã triệt tiêu Heat Stress Overflow & Smart Soil Balancer.")

Đang đọc dữ liệu...
Đang khởi tạo và huấn luyện CTGAN (Epochs=300)...


Gen. (-00.50) | Discrim. (-00.55): 100%|██████████| 300/300 [08:16<00:00,  1.65s/it]


Đang sinh dữ liệu mới...
Đang thực hiện Hậu kiểm Logic (Logic Enforcement)...


C:\Users\AKANEMO\AppData\Local\Temp\ipykernel_17944\675768019.py:67: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[24.75709125 27.44355457 21.78321934 20.18632252 26.46713466 24.39836167
 29.38377422 20.27653189 20.02140512 22.08555192 24.49189881 25.45820198
 15.13752498 20.36828386 23.62504013 25.89138808 26.31637537 24.09256582
 29.94836209 24.7643814  23.12373833  9.61882154 24.57330912 25.59813105
 23.94477348 23.83624557 19.71266711 19.72630994 23.0709135  24.41857086
 29.4651182  28.67489091 19.86758698 28.18706485 19.99968743 29.23796161
 28.39032897 25.02920947 27.34307043 20.42689085 29.71442131 26.04945764
 25.59379166 26.86295702 20.24871176 28.94584648 29.04252445 24.96037079
 27.51466727 22.48915819 25.86387147 22.92884628 20.19888356 25.07841589
 28.29102907 21.74628517 29.68964103 18.09607375 20.52203976 19.76945305
 26.32101126 29.75695472 24.14220451 25.90668786 25.54263711 23.81098285

Hoàn tất! Dữ liệu đã được lưu tại: Agri_Data_CTGAN_20k_Flawless.csv
Phiên bản Flawless: Đã triệt tiêu Heat Stress Overflow & Smart Soil Balancer.
